# TEST_FWD: CLIM-3D Year Mapping and FWD Diagnostics

This notebook collects the CLIM-3D Marina/local year-mapping test in one place. It first checks where the Figure-8 EP-flux correction comparison lives, then uses field fingerprints to infer the Marina-to-local CLIM-3D year mapping, and only after that compares mapped-year FWD.

Default matching feature: daily polar O3 over 60-90N in March-May. The script also supports `--primary-feature u` for daily 10 hPa, 55-75N wind fingerprints; cached U fingerprints are saved after the first run.


## Figure 8 EP-Flux Correction Comparison

Figure 8 comparison files are already generated under `Longrun/Visualization/plots/`:

- `figure8_high_ozone_wind_epfdiv_rm5_no_doubar.png`: legacy EP flux, no `DO_UBAR`, no omega correction.
- `figure8_high_ozone_wind_epfdiv_rm5_doubar_omega.png`: `DO_UBAR` wind-shear/vorticity correction plus omega correction.

<table>
<tr>
<td><b>Figure 8: no DO_UBAR / no omega</b><br><img src="../Visualization/plots/figure8_high_ozone_wind_epfdiv_rm5_no_doubar.png" width="460"></td>
<td><b>Figure 8: DO_UBAR + omega</b><br><img src="../Visualization/plots/figure8_high_ozone_wind_epfdiv_rm5_doubar_omega.png" width="460"></td>
</tr>
</table>

The same low-ozone counterpart is saved as Figure 9:

<table>
<tr>
<td><b>Figure 9: no DO_UBAR / no omega</b><br><img src="../Visualization/plots/figure9_low_ozone_wind_epfdiv_rm5_no_doubar.png" width="460"></td>
<td><b>Figure 9: DO_UBAR + omega</b><br><img src="../Visualization/plots/figure9_low_ozone_wind_epfdiv_rm5_doubar_omega.png" width="460"></td>
</tr>
</table>


## Refresh The Mapping Test

Run this block to regenerate the default O3-fingerprint mapping, FWD comparison tables, and the figures displayed below. It is intentionally a thin notebook wrapper around `fwd_clim3d_feature_mapping_test.py`, so the tested logic remains reusable from the command line.


In [ ]:
import runpy
import sys
from pathlib import Path

DATE_DIR = Path.cwd()
if DATE_DIR.name != "date_treatment":
    DATE_DIR = Path("/home/weiji/restart_exam/code_cleaned/Longrun/date_treatment")
SCRIPT = DATE_DIR / "fwd_clim3d_feature_mapping_test.py"

old_argv = sys.argv[:]
sys.argv = [str(SCRIPT), "--primary-feature", "o3"]
try:
    runpy.run_path(str(SCRIPT), run_name="__main__")
finally:
    sys.argv = old_argv


## Key Tables

These tables are written by the script and are the most useful numbers to quote: Marina merge-history chunks, the inferred mapping, and mapped-year FWD errors.


In [ ]:
import pandas as pd
from pathlib import Path

DATE_DIR = Path.cwd()
if DATE_DIR.name != "date_treatment":
    DATE_DIR = Path("/home/weiji/restart_exam/code_cleaned/Longrun/date_treatment")
OUT = DATE_DIR / "clim3d_feature_mapping_test"

chunks = pd.read_csv(OUT / "marina_merge_history_chunks.csv")
mapping = pd.read_csv(OUT / "feature_matched_year_mapping.csv")
summary = pd.read_csv(OUT / "feature_matched_fwd_summary.csv")
scores = pd.read_csv(OUT / "feature_chunk_sliding_window_scores.csv")

segment_summary = (
    mapping.groupby("chunk_id")
    .agg(
        marina_years=("marina_year", lambda s: f"{int(s.min())}-{int(s.max())}"),
        local_years=("our_year", lambda s: f"{int(s.min())}-{int(s.max())}"),
        marina_run=("marina_run", "first"),
        marina_source_years=("marina_source_year", lambda s: f"{int(s.min())}-{int(s.max())}"),
        match_corr=("chunk_match_correlation", "first"),
        run_match_rate=("run_matches_history", "mean"),
        source_year_match_rate=("source_year_matches_history", "mean"),
    )
    .reset_index()
)

print("Marina merge history:")
display(chunks)
print("Inferred segment mapping:")
display(segment_summary.round(4))
print("FWD summary:")
display(summary[summary["group"].str.contains("50hpa")].round(3))


## Fingerprint Matching Strength

The O3 fingerprint is the default fast test. The U fingerprint result is also shown; it was computed from daily 10 hPa, 55-75N wind and gives the same chunk mapping.

<table>
<tr>
<td><b>O3 fingerprint top windows</b><br><img src="../Visualization/plots/clim3d_feature_mapping_test/clim3d_o3_daily_fingerprint_top_windows.png" width="460"></td>
<td><b>U fingerprint top windows</b><br><img src="../Visualization/plots/clim3d_feature_mapping_test/clim3d_u_fingerprint_top_windows.png" width="460"></td>
</tr>
</table>


## Matched Time Segments

This timeline is the most direct view of the merged-year problem. The upper track is Marina's 200-year merged product; the lower track is the local cleaned 216-year product. Same-colored bars and connectors show the inferred chunk correspondence.

<img src="../Visualization/plots/clim3d_feature_mapping_test/clim3d_feature_matched_segment_timeline.png" width="860">


## Mapped-Year FWD Difference

After fixing the year mapping from physical-field fingerprints, the 50 hPa FWD comparison is nearly one-to-one. The remaining mean offset is about one day, which is consistent with the different local/Marina FWD date indexing conventions rather than a wrong year mapping.

<table>
<tr>
<td><b>50 hPa FWD scatter</b><br><img src="../Visualization/plots/clim3d_feature_mapping_test/clim3d_feature_matched_fwd50_scatter.png" width="430"></td>
<td><b>50 hPa absolute error by pair</b><br><img src="../Visualization/plots/clim3d_feature_mapping_test/clim3d_feature_matched_fwd50_abs_error.png" width="520"></td>
</tr>
</table>


## Marina Native CLIM-3D FWD Profile Check

这个 block 不重新计算 Marina 的 O3 或 FWD，而是直接读取已经生成的 `clim3d_marina_repro_report` 表格。目标是把“用 Marina native CLIM-3D O3 + Marina saved FWD 时可以接近论文 Table 1”的结果放回 `TEST_FWD.ipynb`，和上面的 mapping/FWD 对比放在同一个测试 notebook 里。

图中：灰色是 Marina CLIM-3D 全部年份的 saved FWD profile；蓝色是 Marina rm5 O3 LOW25 年；红色是 Marina rm5 O3 HIGH25 年。50 hPa 处的 LOW25 平均值应接近 Friedel et al. Table 1 的 `27 Apr`。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display

DATE_DIR = Path.cwd()
if DATE_DIR.name != "date_treatment":
    DATE_DIR = Path("/home/weiji/restart_exam/code_cleaned/Longrun/date_treatment")
REPORT_DIR = DATE_DIR / "clim3d_marina_repro_report"
PLOT_DIR = DATE_DIR.parent / "Visualization" / "plots" / "clim3d_marina_repro"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

profile_csv = REPORT_DIR / "marina_saved_fwd_profiles_rm5.csv"
summary_csv = REPORT_DIR / "source_isolation_summary.csv"
if not profile_csv.exists():
    raise FileNotFoundError(profile_csv)
if not summary_csv.exists():
    raise FileNotFoundError(summary_csv)

profiles = pd.read_csv(profile_csv)
summary = pd.read_csv(summary_csv)
focus = summary[
    summary["test"].isin([
        "paper Table 1 reference",
        "Marina native saved FWD + Marina O3 rm5_from_raw",
        "Our native FWD + our O3 rm5",
        "Mapped pair: Marina FWD + our O3 rm5",
        "Mapped pair: our FWD + Marina O3 rm5_from_raw",
    ])
].copy()

MONTH_START_DOY = np.array([1, 32, 60, 91, 121, 152, 182], dtype=float)
MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul"]

def _doy_to_month_day(doy):
    doy = int(round(float(doy)))
    month_lengths = [31, 28, 31, 30, 31, 30, 31]
    month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul"]
    for name, nday in zip(month_names, month_lengths):
        if doy <= nday:
            return f"{name} {doy:02d}"
        doy -= nday
    return "Jul+"

def _month_formatter(x, _pos=None):
    if not np.isfinite(x):
        return ""
    return _doy_to_month_day(x)

fig, ax = plt.subplots(figsize=(6.6, 6.2), dpi=160)
ax.plot(profiles["all_mean_doy"], profiles["plev_hpa"], color="0.35", lw=2.0, label="All Marina CLIM-3D years")
ax.plot(profiles["low25_mean_doy"], profiles["plev_hpa"], color="navy", lw=2.4, label="Marina rm5 LOW25 O3 years")
ax.plot(profiles["high25_mean_doy"], profiles["plev_hpa"], color="firebrick", lw=2.4, label="Marina rm5 HIGH25 O3 years")

p50 = profiles.iloc[(profiles["plev_hpa"] - 50).abs().argmin()]
ax.scatter([p50["low25_mean_doy"]], [p50["plev_hpa"]], s=62, color="navy", edgecolor="white", zorder=5)
ax.axhline(50, color="0.55", lw=0.9, ls=":")
ax.axvline(117.0, color="k", lw=1.0, ls="--", alpha=0.75, label="Paper Table 1: 27 Apr")
ax.annotate(
    f"50 hPa LOW25 = {_doy_to_month_day(p50['low25_mean_doy'])}\n({p50['low25_mean_doy']:.2f} DOY)",
    xy=(p50["low25_mean_doy"], p50["plev_hpa"]),
    xytext=(p50["low25_mean_doy"] + 6, 35),
    textcoords="data",
    arrowprops=dict(arrowstyle="->", lw=0.9, color="navy"),
    fontsize=9,
    color="navy",
)

ax.set_yscale("log")
ax.set_ylim(70, 0.08)
ax.set_yticks([0.1, 0.5, 1, 5, 10, 30, 50])
ax.set_yticklabels(["0.1", "0.5", "1", "5", "10", "30", "50"])
ax.set_xlim(100, 126)
ax.xaxis.set_major_formatter(FuncFormatter(_month_formatter))
ax.set_xlabel("Final warming date")
ax.set_ylabel("Pressure (hPa)")
ax.set_title("Marina native CLIM-3D saved FWD selected by Marina rm5 O3", fontsize=11, weight="bold")
ax.grid(True, which="both", color="0.86", lw=0.6)
ax.legend(loc="lower left", fontsize=8, frameon=True)
fig.tight_layout()

stem = "clim3d_marina_native_saved_fwd_profiles_rm5_test_fwd"
for suffix in ["png", "svg", "pdf"]:
    fig.savefig(PLOT_DIR / f"{stem}.{suffix}", dpi=300 if suffix == "png" else None, bbox_inches="tight")
plt.show()

print(f"[WRITE] {PLOT_DIR / (stem + '.png')}")
print("Source-isolation rows most relevant for the TEST_FWD conclusion:")
display(focus[["test", "fwd_source", "o3_source_for_low25", "candidate_rows", "n_low25", "mean_low25_50hpa_doy", "mean_low25_50hpa_date", "delta_vs_paper_days"]])


## Optional: Re-run The U-Wind Fingerprint

The U-wind fingerprint is more expensive if the cache is absent because it reads the yearly 3D U files. If `local_u10_55_75N_days1_181.npy` and `marina_u10_55_75N_days1_181.npy` already exist, this is quick.


In [ ]:
# Optional heavier check. Uncomment to regenerate U-fingerprint mapping.
# import runpy, sys
# from pathlib import Path
# DATE_DIR = Path.cwd() if Path.cwd().name == "date_treatment" else Path("/home/weiji/restart_exam/code_cleaned/Longrun/date_treatment")
# SCRIPT = DATE_DIR / "fwd_clim3d_feature_mapping_test.py"
# old_argv = sys.argv[:]
# sys.argv = [str(SCRIPT), "--primary-feature", "u"]
# try:
#     runpy.run_path(str(SCRIPT), run_name="__main__")
# finally:
#     sys.argv = old_argv


## Output Files

- Mapping CSV: `clim3d_feature_mapping_test/feature_matched_year_mapping.csv`
- FWD pair CSV: `clim3d_feature_mapping_test/feature_matched_fwd50_by_pair.csv`
- FWD summary CSV: `clim3d_feature_mapping_test/feature_matched_fwd_summary.csv`
- Figure-8 inventory CSV: `clim3d_feature_mapping_test/figure8_epflux_variant_inventory.csv`
